**Machine Learning Model Development**

This notebook develops and compares several machine learning models for predicting road traffic collision severity.

The main objectives are:

- Load the processed machine learning dataset.
- Prepare the data for modelling.
- Split the dataset into training and testing sets.
- Build preprocessing and machine learning pipelines.
- Train multiple classification models.
- Compare model performance using cross-validation.
- Select the best-performing model for deployment.

Import Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report
from time import perf_counter

In [2]:
df = pd.read_csv('/content/drive/MyDrive/UK-Accident-Prediction/model_dataset.csv',
                 dtype={'collision_index': 'string'})

In [3]:
X = df.drop(columns=['collision_index', 'collision_severity'])
y = df['collision_severity']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42,stratify=y)

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42,stratify=y_train)

In [6]:
X_train_sample, _, y_train_sample, _ = train_test_split(X_train, y_train, train_size=100_000, random_state=42,stratify=y_train)

In [7]:
print(X_train_sample.shape, X_val.shape, X_test.shape)
print(y_train_sample.shape, y_val.shape, y_test.shape)

(100000, 34) (301880, 34) (377350, 34)
(100000,) (301880,) (377350,)


In [8]:
numerical_features = [
    'collision_year',
    'location_easting_osgr',
    'location_northing_osgr',
    'longitude',
    'latitude',
    'number_of_vehicles',
    'number_of_casualties',
    'speed_limit',
    'average_driver_age',
    'average_vehicle_age',
    'max_vehicle_age',
    'average_casualty_age',
    'max_casualty_age',
    'month',
    'day',
    'hour'
]

In [9]:
categorical_features = [
    column for column in X_train_sample.columns if column not in numerical_features]

In [10]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [11]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', numerical_pipeline, numerical_features),
        ('categorical', categorical_pipeline, categorical_features)
    ], sparse_threshold=1.0)

In [13]:
X_train_processed = preprocessor.fit_transform(X_train_sample)
X_val_processed = preprocessor.transform(X_val)

In [14]:
X_train_processed.shape

(100000, 580)

In [15]:
models = {
    'Logistic Regression': LogisticRegression(
        solver='saga',
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),

    'Decision Tree': DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced_subsample',
        n_jobs=-2,
        random_state=42
    )
}

In [16]:
results_rows = []

In [17]:
for name, model in models.items():
  start = perf_counter()

  model.fit(X_train_processed, y_train_sample)

  training_seconds = perf_counter() - start
  predictions = model.predict(X_val_processed)

  results_rows.append({
      'Model': name,
      'Validation Accuracy': accuracy_score(y_val, predictions),
      'Validation Macro F1': f1_score(y_val, predictions, average='macro'),
      'Training Time (s)': training_seconds
  })

  print(f'\n{name}')
  print(classification_report(y_val, predictions,
                              digits=3, zero_division=0))

  results = pd.DataFrame(results_rows).sort_values('Validation Macro F1',
                                              ascending=False)

  results


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Logistic Regression
              precision    recall  f1-score   support

           1      0.039     0.579     0.073      3878
           2      0.263     0.410     0.320     51567
           3      0.898     0.598     0.718    246435

    accuracy                          0.566    301880
   macro avg      0.400     0.529     0.371    301880
weighted avg      0.779     0.566     0.642    301880


Decision Tree
              precision    recall  f1-score   support

           1      0.038     0.452     0.069      3878
           2      0.243     0.433     0.311     51567
           3      0.889     0.589     0.709    246435

    accuracy                          0.561    301880
   macro avg      0.390     0.491     0.363    301880
weighted avg      0.768     0.561     0.633    301880


Random Forest
              precision    recall  f1-score   support

           1      0.043     0.535     0.079      3878
           2      0.286     0.405     0.335     51567
           3      0.892 

In [17]:
df = pd.read_csv(
    '/content/drive/MyDrive/UK-Accident-Prediction/model_dataset.csv',
    dtype={'collision_index': 'string'}
)

In [18]:
training_data = df.drop(
    index=X_val.index.union(X_test.index))

In [19]:
X_train_sample, _, y_train_sample, _ = train_test_split(
    training_data.drop(
        columns=['collision_index','collision_severity']),
    training_data['collision_severity'],
    train_size=400_000,
    random_state=42,
    stratify=training_data['collision_severity'])

In [20]:
from sklearn.base import clone

In [22]:
candidate_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', clone(models['Random Forest']))
])

start = perf_counter()

candidate_pipeline.fit(X_train_sample, y_train_sample)

print(perf_counter() - start)

184.4551585969998


In [23]:
predictions = candidate_pipeline.predict(X_val)

print('Validation Accuracy:', accuracy_score(y_val, predictions))
print('Validation Macro F1:', f1_score(y_val, predictions, average='macro'))
print(classification_report(y_val, predictions, digits=3, zero_division=0))

Validation Accuracy: 0.5858056181264079
Validation Macro F1: 0.3791683334312507
              precision    recall  f1-score   support

           1      0.040     0.648     0.076      3878
           2      0.286     0.366     0.321     51567
           3      0.896     0.631     0.740    246435

    accuracy                          0.586    301880
   macro avg      0.408     0.548     0.379    301880
weighted avg      0.781     0.586     0.660    301880



In [24]:
production_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', models['Random Forest'])
])

production_pipeline.set_params(model__n_jobs=1)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(sparse_threshold=1.0,
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['collision_year',
                                                   'location_easting_osgr',
                                                   'location_northing_osgr',
                                                   'longitude', 'latitude',
                                                   'number_of_vehicles',
                                                   'number_of_casualties',
                                                   'speed_limit',
                                                   'average_driv...
                                                   'light_conditions',
                                                   'weather_conditions',
                                                   'road_surface_conditions',
                                                   'special_conditions_at_site',
                                                   'carriageway_hazards',
                                                   'urban_or_rural_area',
                                                   'did_police_officer_attend_scene_of_accident',
                                                   'trunk_road_flag',
                                                   'most_common_vehicle_type'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=12, min_samples_leaf=10,
                                        n_jobs=1, random_state=42))])

In [25]:
test_predictions = production_pipeline.predict(X_test)

print('Test Accuracy:', accuracy_score(y_test, test_predictions))
print('Test Macro F1:', f1_score(y_test, test_predictions, average='macro'))
print(classification_report(y_test, test_predictions, digits=3, zero_division=0))

Test Accuracy: 0.6098423214522327
Test Macro F1: 0.39093163993064334
              precision    recall  f1-score   support

           1      0.044     0.547     0.081      4848
           2      0.288     0.408     0.338     64459
           3      0.893     0.653     0.754    308043

    accuracy                          0.610    377350
   macro avg      0.408     0.536     0.391    377350
weighted avg      0.778     0.610     0.674    377350



In [26]:
import joblib
import os

In [27]:
model_path = (
    '/content/drive/MyDrive/UK-Accident-Prediction/'
    'production_pipeline.joblib'
)

In [28]:
joblib.dump(production_pipeline, model_path, compress=3)

['/content/drive/MyDrive/UK-Accident-Prediction/production_pipeline.joblib']

In [29]:
print('Saved model:', model_path)
print('Model size:', os.path.getsize(model_path) / 1024**2)

Saved model: /content/drive/MyDrive/UK-Accident-Prediction/production_pipeline.joblib
Model size: 3.4259109497070312


In [30]:
import sklearn
import numpy
import scipy

In [31]:
print('scikit-learn version:', sklearn.__version__)
print('numpy version:', numpy.__version__)
print('scipy version:', scipy.__version__)
print('pandas version:', pd.__version__)
print('joblib version:', joblib.__version__)

scikit-learn version: 1.6.1
numpy version: 2.1.3
scipy version: 1.16.3
pandas version: 2.2.3
joblib version: 1.6.0


In [32]:
candidate_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', clone(models['Random Forest']))
])

candidate_pipeline.set_params(model__max_depth=16)

candidate_pipeline.fit(X_train_sample, y_train_sample)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(sparse_threshold=1.0,
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['collision_year',
                                                   'location_easting_osgr',
                                                   'location_northing_osgr',
                                                   'longitude', 'latitude',
                                                   'number_of_vehicles',
                                                   'number_of_casualties',
                                                   'speed_limit',
                                                   'average_driv...
                                                   'light_conditions',
                                                   'weather_conditions',
                                                   'road_surface_conditions',
                                                   'special_conditions_at_site',
                                                   'carriageway_hazards',
                                                   'urban_or_rural_area',
                                                   'did_police_officer_attend_scene_of_accident',
                                                   'trunk_road_flag',
                                                   'most_common_vehicle_type'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=16, min_samples_leaf=10,
                                        n_jobs=1, random_state=42))])

In [33]:
import io
import joblib

predictions = candidate_pipeline.predict(X_val)

print('Training samples:', len(y_train_sample))
print('Validation Accuracy:', accuracy_score(y_val, predictions))
print(
    'Validation Macro F1:',
    f1_score(y_val, predictions, average='macro')
)

print(classification_report(
    y_val,
    predictions,
    digits=3,
    zero_division=0
))

with io.BytesIO() as buffer:
    joblib.dump(candidate_pipeline, buffer, compress=3)
    print('Model size (MB):', buffer.tell() / 1024**2)

Training samples: 400000
Validation Accuracy: 0.6118523916788128
Validation Macro F1: 0.39451701573950176
              precision    recall  f1-score   support

           1      0.046     0.550     0.085      3878
           2      0.288     0.427     0.344     51567
           3      0.897     0.652     0.755    246435

    accuracy                          0.612    301880
   macro avg      0.410     0.543     0.395    301880
weighted avg      0.782     0.612     0.676    301880

Model size (MB): 14.780120849609375


In [34]:
production_pipeline = candidate_pipeline
production_pipeline.set_params(model__n_jobs=1)

test_predictions = production_pipeline.predict(X_test)

print('Test Accuracy:', accuracy_score(y_test, test_predictions))
print('Test Macro F1:', f1_score(y_test, test_predictions, average='macro'))

print(classification_report(y_test, test_predictions, digits=3, zero_division=0))

joblib.dump(production_pipeline, '/content/drive/MyDrive/UK-Accident-Prediction/production_pipeline.joblib', compress=3)

print('Production pipeline saved.')

Test Accuracy: 0.6127679872797138
Test Macro F1: 0.396002342806862
              precision    recall  f1-score   support

           1      0.047     0.564     0.087      4848
           2      0.290     0.429     0.346     64459
           3      0.897     0.652     0.755    308043

    accuracy                          0.613    377350
   macro avg      0.411     0.548     0.396    377350
weighted avg      0.783     0.613     0.677    377350

Production pipeline saved.


In [39]:
depth_candidate = clone(production_pipeline)

depth_candidate.set_params(
    model__max_depth=20,
    model__n_jobs=2
)

depth_candidate.fit(X_train_sample, y_train_sample)

predictions = depth_candidate.predict(X_val)

print('Validation Accuracy:', accuracy_score(y_val, predictions))
print(
    'Validation Macro F1:',
    f1_score(y_val, predictions, average='macro')
)

print(classification_report(
    y_val,
    predictions,
    digits=3,
    zero_division=0
))

Validation Accuracy: 0.6439280508811448
Validation Macro F1: 0.41089740272738623
              precision    recall  f1-score   support

           1      0.054     0.459     0.097      3878
           2      0.295     0.460     0.360     51567
           3      0.895     0.685     0.776    246435

    accuracy                          0.644    301880
   macro avg      0.415     0.535     0.411    301880
weighted avg      0.782     0.644     0.696    301880



In [40]:
import io
import joblib

with io.BytesIO() as buffer:
    joblib.dump(depth_candidate, buffer, compress=3)
    print('Model size (MB):', buffer.tell() / 1024**2)

Model size (MB): 26.372928619384766


In [41]:
production_pipeline = depth_candidate
production_pipeline.set_params(model__n_jobs=1)

test_predictions = production_pipeline.predict(X_test)

print('Test Accuracy:', accuracy_score(y_test, test_predictions))
print(
    'Test Macro F1:',
    f1_score(y_test, test_predictions, average='macro')
)

print(classification_report(
    y_test,
    test_predictions,
    digits=3,
    zero_division=0
))

joblib.dump(
    production_pipeline,
    '/content/drive/MyDrive/UK-Accident-Prediction/production_pipeline.joblib',
    compress=3
)

print('Production pipeline saved.')

Test Accuracy: 0.6448469590565787
Test Macro F1: 0.41331723846859764
              precision    recall  f1-score   support

           1      0.056     0.480     0.101      4848
           2      0.298     0.463     0.362     64459
           3      0.895     0.685     0.776    308043

    accuracy                          0.645    377350
   macro avg      0.416     0.543     0.413    377350
weighted avg      0.782     0.645     0.697    377350

Production pipeline saved.


## Model Selection Summary

- Selected a Random Forest trained on 400,000 records.
- Used 100 trees, maximum depth 20 and minimum leaf size 10.
- Used balanced subsample class weights.
- Validation Accuracy: 0.6439.
- Validation Macro F1: 0.4109.
- Test Accuracy: 0.6448.
- Test Macro F1: 0.4133.
- Compressed pipeline size: 26.37 MB.
- Minority-class recall improved over the original project, while accuracy and Macro F1 remained lower.
- The test set was evaluated during multiple iterations.